# 第 10 课 · 为什么股价跌不成负数（浏览器版）

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JavenZheng/monte-carlo-finance/blob/main/notebooks/10-normal-and-lognormal.ipynb)

配套课文：[第 10 课 · 为什么股价跌不成负数（网页版）](../docs/m0-foundations/10-normal-and-lognormal.md)。

**怎么用**：本地装了 Jupyter 就直接打开这个文件；不想装任何东西，打开 [https://jupyter.org/try](https://jupyter.org/try)（或 Colab），把这一页整份放进去，每一格按 `Shift + Enter` 从上往下跑。

只用 Python 标准库（`random` 和 `math`），不需要安装任何东西。

**每格跑出来的数字和课文里印的逐字一致**（种子钉在 7）。有几格要跑一万条、每条 250 天的连乘，单格大约两三秒，别着急。

## 一、你的模拟程序造出了一个不该存在的东西

上一课你会造任何形状了，于是你可能顺手试了最自然的那个动作：**模拟一只股票。**

你手上有个 3000 块的基金，你觉得它每天大概涨跌 1% 左右。你按最直的思路写：今天的价格等于昨天的价格加上一个随机的涨跌。为了看数方便，我们一律把起点记成 100（这样每个结果都是百分比，3000 块那只只要把数乘 30 就行）。

跑一万遍，最惨的一遍是 **−208.5**，一万遍里有 **1036 遍**的终点是负数（这一格跑到第四节就会真的跑给你看）。

一只基金的价格不可能是负的。你检查一下代码，逻辑没写错、随机数没写错、参数也合理。那问题出在哪？

问题出在你把"涨跌"当成了**几块钱**。而这门课要说的第一件事是：**价格上的涨跌其实是按比例的，比例和块钱不是同一种东西。** 这一件事讲透，负价格自动消失，而且你会顺手拿到金融里最常用的那个分布。

## 二、先跌一半再涨一半，你回不到原点

在写任何代码之前，先算三道一年级算术。

你手上 100 块。第一天跌一半，剩 50。第二天涨一半——是 50 的一半，25——回到 75。

$$100 \times 0.5 \times 1.5 = 75.0$$

换成先涨后跌也一样：100 涨一半到 150，再跌一半（150 的一半是 75），回到 75。

而如果是块钱呢？+50 块钱、−50 块钱，回到 100。跑一遍程序，三种写法的结果分别是 **75.0、75.0、100**（前两个数字是同一件事的两种顺序）：

In [1]:
print(f"先跌 50% 再涨 50%：100 × 0.5 × 1.5 = {100 * 0.5 * 1.5:.1f}")
print(f"先涨 50% 再跌 50%：100 × 1.5 × 0.5 = {100 * 1.5 * 0.5:.1f}")
print(f"而 ±50 块钱来回：100 + 50 − 50 = {100 + 50 - 50}")

先跌 50% 再涨 50%：100 × 0.5 × 1.5 = 75.0
先涨 50% 再跌 50%：100 × 1.5 × 0.5 = 75.0
而 ±50 块钱来回：100 + 50 − 50 = 100


这就是全部的秘密：**块钱可以抵消，比例不能抵消。** 跌掉的那一刀是按小本金算的，涨回来的那一刀也是按小本金算的，两头不对称。

**记住这两行的差别，第四节的负价格和第五节的"越晃越亏"都从它长出来。**

## 三、把第 08 课那个钟形拿来当每天的涨跌

第 08 课你已经见过"中间鼓、两头瘦"那个形状，它叫正态分布。现在我们要用它，只需要一行工具：

```python
random.gauss(平均, 起伏)
```

它给你**一个**服从正态分布的数：以"平均"为中心，散布宽度大约是"起伏"。这就是第 09 课那条一米队伍的正态版本——你不用再切段，直接要一个钟形里的抽签。

于是"模拟价格"有两种写法，差别就是第二节那两道算术：

```python
p += random.gauss(0, 5)          # 每天加减 5 块钱
p *= 1 + random.gauss(0, 0.02)   # 每天涨跌 2%
```

第一种是块钱，第二种是比例。把每一段前面都钉上种子 `random.seed(7)`，各跑一万遍，一年按 250 天算。

下面这一格把这两条路各自写成一台机器（`add_price` 是块钱版，`mult_price` 是比例版），外加两个小工具：`quantile` 取排在某个位置的数，`histogram` 把一列数排成字符图。这一格只是定义，运行它不会打印任何东西：

In [2]:
import math
import random


def add_price(start, daily_sd, days, runs):
    """每天的涨跌当成"几块钱"，正态，直接加。返回每条路径的终值和中途最低点。"""
    ends = []
    lows = []
    for _ in range(runs):
        p = start
        low = start
        for _ in range(days):
            p += random.gauss(0, daily_sd)
            if p < low:
                low = p
        ends.append(p)
        lows.append(low)
    return ends, lows


def mult_price(start, mean_r, sd_r, days, runs):
    """每天的涨跌当成"百分之几"，正态，乘上去。返回每条路径的终值。"""
    out = []
    for _ in range(runs):
        p = start
        for _ in range(days):
            p *= 1 + random.gauss(mean_r, sd_r)
        out.append(p)
    return out


def quantile(sorted_vals, q):
    return sorted_vals[int(q * len(sorted_vals))]


def histogram(values, bins=9, width=44):
    lo = min(values)
    hi = max(values)
    step = (hi - lo) / bins
    counts = [0] * bins
    for x in values:
        k = int((x - lo) / step)
        if k == bins:
            k = bins - 1
        counts[k] = counts[k] + 1
    biggest = max(counts)
    for i in range(bins):
        left = lo + i * step
        bar = "*" * max(1, int(round(counts[i] / biggest * width)))
        print(f"{left:7.0f}  {bar}  {counts[i]}")

## 四、块钱那版会给你负价格，比例那版不会

**加减 5 块钱那版**（起点 100，平均 0）：

In [3]:
random.seed(7)
adds, lows = add_price(100, 5, 250, 10000)
neg = 0
for v in adds:
    if v < 0:
        neg += 1
ever = 0
for v in lows:
    if v < 0:
        ever += 1
adds.sort()
print("起点 100，每天 ±5 块钱乱晃（平均 0），一年 250 天，跑一万遍：")
print(f"  终值是负数的：{neg} 遍，占 {neg / len(adds) * 100:.1f}%")
print(f"  中途跌破过零的：{ever} 遍，占 {ever / len(lows) * 100:.1f}%")
print(f"  最惨的一遍：{adds[0]:.1f} 元")
print(f"  一成低于 {quantile(adds, 0.1):.0f}，中位 {quantile(adds, 0.5):.0f}，九成不超过 {quantile(adds, 0.9):.0f}")

起点 100，每天 ±5 块钱乱晃（平均 0），一年 250 天，跑一万遍：
  终值是负数的：1036 遍，占 10.4%
  中途跌破过零的：1948 遍，占 19.5%
  最惨的一遍：-208.5 元
  一成低于 -2，中位 100，九成不超过 201


一万遍里有整整一成遍，一年后这只基金值负钱。你的程序没坏，它只是**忠实地演了一个你把世界搭错了的杯子**（第 01 课第六节那句话在这里第一次真的用上）。

**涨跌 2% 那版**（同样起点 100、平均 0）：

In [4]:
random.seed(7)
mults = mult_price(100, 0.0, 0.02, 250, 10000)
mults.sort()
print("起点 100，每天平均涨 0%、起伏 2%，一年 250 天，跑一万遍：")
print(f"  最惨的一遍：{mults[0]:.2f} 元（没有一遍是负的）")
print(f"  一成低于 {quantile(mults, 0.1):.1f}，中位 {quantile(mults, 0.5):.1f}，九成不超过 {quantile(mults, 0.9):.1f}")
print(f"  一万遍的平均：{sum(mults) / len(mults):.1f}")
below = 0
for v in mults:
    if v < 100:
        below += 1
print(f"  一年后比起点还低的：占 {below / len(mults) * 100:.1f}%")

起点 100，每天平均涨 0%、起伏 2%，一年 250 天，跑一万遍：
  最惨的一遍：27.63 元（没有一遍是负的）
  一成低于 63.3，中位 95.3，九成不超过 142.3
  一万遍的平均：99.9
  一年后比起点还低的：占 56.3%


**价格永远为正**——因为它只能被越乘越小，乘不到零以下去。这就是"股价只能为正"这件事的全部工程实现：不是加了什么限制，是**乘**这个动作本身就不会穿过零。

再看形状，把一万条路径的终值排成一张图（`*` 的长短是次数）：

```
     28  *******  601
     58  *******************************************  3491
     89  ********************************************  3569
    119  ********************  1601
    150  *******  542
    180  **  133
    211  *  51
    241  *  8
    272  *  4
```

In [5]:
print("一万条路径的终值排成什么形状：")
histogram(mults)

一万条路径的终值排成什么形状：
     28  *******  601
     58  *******************************************  3491
     89  ********************************************  3569
    119  ********************  1601
    150  *******  542
    180  **  133
    211  *  51
    241  *  8
    272  *  4


跟第 08 课那张钟比：**左边到 28 就断了（不能再低），右边一路拖到 272 还没断。** 这个"有底、无顶"的形状叫**对数正态**——它是正态的按比例版本，也是金融里用得最多的一个分布。你现在只需要认得它的长相：左边削平，右边拖尾。

## 五、"平均不涨"不等于"你不会亏"

第四部分那一串数里，有三个必须放在一起看：

- 每天平均涨跌 **0%**；
- 一万遍的平均终值是 **99.9**，也就是说不涨不跌，公平游戏；
- 可一年后**比本金还低的路径占 56.3%**，中位数只有 **95.3**。

**平均是 100，典型却是 95，超过一半的人亏钱。**

这不是矛盾，就是第二节那道算术在大量天数上累积起来的结果：每一天"跌 2%"和"涨 2%"看起来对称，但跌的那刀砍在更大的本金上。晃来晃去，钱就自己往下走了一点。

把起伏加大，这件事会变得非常刺眼。下面两组**平均涨跌完全一样**（每天 +0.1%），只有起伏不同，各跑一万遍：

In [6]:
random.seed(7)
calm = mult_price(100, 0.001, 0.02, 250, 10000)
calm.sort()
random.seed(7)
wild = mult_price(100, 0.001, 0.04, 250, 10000)
wild.sort()
print("两个都是每天平均涨 0.1%，一年 250 天，各跑一万遍：")
print(f"  起伏 2%：中位 {quantile(calm, 0.5):.1f}，平均 {sum(calm) / len(calm):.1f}，亏钱的占 {sum(1 for v in calm if v < 100) / len(calm) * 100:.1f}%")
print(f"  起伏 4%：中位 {quantile(wild, 0.5):.1f}，平均 {sum(wild) / len(wild):.1f}，亏钱的占 {sum(1 for v in wild if v < 100) / len(wild) * 100:.1f}%")

两个都是每天平均涨 0.1%，一年 250 天，各跑一万遍：
  起伏 2%：中位 122.4，平均 128.2，亏钱的占 26.2%
  起伏 4%：中位 105.6，平均 127.8，亏钱的占 46.7%


**平均值几乎没动（128.2 → 127.8），典型结果掉了 17 块钱（122.4 → 105.6），亏钱的概率从四分之一跳到将近一半。**

同一把尺子画成两格，就看得见那个"往左跑"：黑实线（中位）在右格里明显偏左，灰虚线（平均）却几乎没动；橙色那一块是"比本金低"的面积。

![两格一年后被本金值多少的直方图，横轴同一把尺子：起伏 2% 那格中位 122.4、平均 128.2、比本金低占 26.2%；起伏 4% 那格中位 105.6、平均 127.8、比本金低占 46.7%](https://raw.githubusercontent.com/JavenZheng/monte-carlo-finance/main/assets/fig10_volatility_drag.svg)

图由 [code/fig10_volatility_drag.py](../code/fig10_volatility_drag.py) 生成，用的就是上面这一轮抽签（种子 7、各一万遍）。

再把起伏拉到 10%、平均涨跌归零：中位是 **28.4**，平均是 **97.3**——按平均算几乎没亏，按你会遇到的那一次算，本金剩不到三成。

In [7]:
random.seed(7)
flat10 = mult_price(100, 0.0, 0.10, 250, 10000)
flat10.sort()
print(f"再把起伏拉到 10%、平均涨 0%：中位 {quantile(flat10, 0.5):.1f}，平均 {sum(flat10) / len(flat10):.1f}")
print(f"  理论给的中位：100 × e^(−0.10² ÷ 2 × 250) = {100 * math.exp(-0.10 ** 2 / 2 * 250):.1f}")

再把起伏拉到 10%、平均涨 0%：中位 28.4，平均 97.3
  理论给的中位：100 × e^(−0.10² ÷ 2 × 250) = 28.7


这就是"起伏"这个词在按比例的世界里真正的价格：**同样是"平均涨 0.1%"，晃得越凶，你个人拿到手的结果越差，而报表上的平均值对此毫无反应。** 第 05 课那句"平均会骗你"在这里换了一副面孔回来：不是因为有个爆款把平均拉高，而是因为**平均是加法的孩子，你的账户是乘法的**。

## 六、log：把 250 个乘法变成 250 个加法

上面那套道理还差一件趁手工具。我们想让第 07、08 课那些只适用于**加法**的算式（毛边 = 起伏 ÷ √条数、乘二换成九成五）也用在价格上，但价格是连乘的。

`math.log` 干的就是这件事：**它把一串乘法变成一串加法。** 拿一条真实路径验证（起伏 2%、平均每天 +0.1%，250 天）：

In [8]:
random.seed(7)
one = [random.gauss(0.001, 0.02) for _ in range(250)]
p = 100.0
for r in one:
    p *= 1 + r
direct = math.log(p / 100)
added = 0
for r in one:
    added += math.log(1 + r)
print(f"一条路径：连乘完再取 log = {direct:.4f}；把每天的 log(1+r) 加起来 = {added:.4f}")
print("同一个数。所以日收益近似正态时，'log(终值 ÷ 起点)'就是那个钟形。")
theory = 250 * 0.001 - 0.02 ** 2 / 2 * 250
print(f"起伏 2%、平均每天涨 0.1%，一年的中位理论值：100 × e^{theory:.3f} = {100 * math.exp(theory):.1f}")

一条路径：连乘完再取 log = 0.4325；把每天的 log(1+r) 加起来 = 0.4325
同一个数。所以日收益近似正态时，'log(终值 ÷ 起点)'就是那个钟形。
起伏 2%、平均每天涨 0.1%，一年的中位理论值：100 × e^0.200 = 122.1


同一个数，两种算法。所以你真正该假设"每天近似正态"的，不是价格本身，而是**每天那个 `log(1+涨跌)`**。这一句就是原书模拟股价时"相加的不是价格而是收益率"的全部理由。

变成加法以后，第五节那件事有个干净的式子。它替你回答的问题是：**"每天平均涨 0.1%、起伏 2%，一年下来我典型能拿到多少？"**

$$\text{中位终值} \approx \text{起点} \times e^{T \times \text{日均涨跌} - \frac{\sigma^2}{2} T}$$

代进去（$T = 250$、日均 $0.001$、$\sigma = 0.02$）：$100 \times e^{0.25 - 0.05} = 122.1$，而模拟跑出来是 **122.4**。又是两个独立算法撞到同一个数——和第四、第五节一样，式子和抽签互相作证。

式子里那个 $-\frac{\sigma^2}{2}$ 就是第五节掉下去的那 17 块钱：**起伏越大，它减得越多**（$\sigma$ 从 2% 变成 4%，这一项就变成四倍，因为它是平方）。原书 **§3.2 几何布朗运动（书页 93）** 那个连续版本 $S_T = S_0 \exp\left((r - \frac{\sigma^2}{2})T + \sigma W_T\right)$ 里，$\sigma^2/2$ 就明晃晃写在指数上，你今天在模拟里看见它干活了。

## 七、今晚就能用，但先说三件事

代码在 [code/lesson10_normal_and_lognormal.py](../code/lesson10_normal_and_lognormal.py)。你要改的是三个参数：起点、日均涨跌、起伏（`mult_price(3000, 0.0005, 0.015, 250, 10000)` 就是"我那只三千万……三千块的基金"）。

三件可以直接拿去用的事：

**给你的持仓做一次"最坏情况体检"。** 报三个数而不是一个数：一成低于多少、中位多少、九成不超过多少。以 3000 块起、日均 +0.05%、起伏 1.5% 为例，实测是 **2435 / 3310 / 4470**，另有 **34.0%** 的概率一年后比 3000 还低。这句话可以原样进你的家庭财务讨论。

**当别人拿"期望收益"说服你时，问一句"典型情况呢"。** 第五节那两组数（平均 128.2 对 127.8、中位 122.4 对 105.6）就是这一问的标准杀伤力：同一个平均，两种人生。

**把你自己的任何模拟做一次"负数体检"。** 你的模型如果会造出负的价格、负的销量、负的人数，那你就是把块钱和比例搞混了——这是最容易发现、也最容易被忽略的建模错误。**能跑出负数的价格模型，一定错。**

然后三件事必须说清楚：

**第一，正态会低估你最怕的那种日子。** 钟形两头瘦得极快，真实市场却会有一天跌 10% 的事反复发生。上面这一万遍里最差的路径是 −72%，听起来很惨，但真实市场不会给你这种形状的尾巴（第 05 课的长尾、原书 **§9.3 厚尾设定（书页 506）** 一整节就在处理这件事）。用正态做估计，你的"最坏情况"通常偏乐观。

**第二，"每天独立地涨跌"是个很强的假设。** 我们让每天的抽签互不影响（第 01 课那个"抽完必须放回"的规矩），可真实的涨跌会扎堆：大跌之后紧跟大跌的概率比独立假设给的高。这一条不影响你今天算数，但它是你以后读到原书第 3 章、第 6 章时会重新遇到的东西。

**第三，日均涨跌和起伏这两个参数从哪来，比模型本身更要命。** 用过去一年的记录估出来的这两个数，本身就带第 07 课那种毛边（`random.gauss` 的两个参数写错了，一万遍会精确地告诉你一个错答案，就像第 01 课那个杯子）。

## 这一课带走的一句话

涨跌是按比例的，所以乘法造出来的价格永远为正、右边拖尾；也正因为是乘法，"平均不涨"的东西对中位数是往下掉的——起伏在指数上偷偷扣你 $\sigma^2/2$。

配套课文：[第 10 课 · 为什么股价跌不成负数（网页版）](../docs/m0-foundations/10-normal-and-lognormal.md)。

---

**许可**：正文 CC BY-NC-SA 4.0，代码 MIT，见 [LICENSE.md](../LICENSE.md)。

**建议购买正版**：Paul Glasserman, *Monte Carlo Methods in Financial Engineering*, Springer, 2003；中译本《金融工程中的蒙特卡罗方法》。本 notebook 未复制原书任何文字、图表或表格。
